# Qwen3.5-4B IR4 test (vLLM dual-GPU)

This independent comparison reuses the existing IR8 cache and selects frames 2, 4, 6, and 8. Create the package locally, copy its printed `manifest_sha256` into `EXPECTED_MANIFEST_SHA256` in the first code cell, and attach that exact ZIP or extracted package as a private Kaggle input. The digest must come from a trusted local build.

Test inference runs on **vLLM 0.19.1 with data parallelism across two T4s**. Each replica has a scheduler limit of 16 sequences; the Runner submits a global batch of 32 and splits it into 16 requests per replica. Both replica processes start before either model load is awaited, so the DP model initialization overlaps. The GPUs have no NVLink, so independent replicas also avoid TP all-reduce on every forward pass. Set `PARALLEL_MODE = "tp"` to reproduce the earlier TP configuration. The signed run contract records both per-replica and Runner batch limits.

The smoke cell also asserts that worker GPU memory telemetry is non-empty and that the prefix-cache hit rate is measured rather than missing.

Use a Kaggle 2x T4 session and run the smoke check before complete test inference.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
BUNDLE_INPUT = None
EXPECTED_MANIFEST_SHA256 = None  # paste manifest_sha256 from the trusted package command  # ZIP，或含 qwen35_bundle_manifest.json 的已解压目录
WEIGHTS_INPUT = None  # 可选：含 cuhkx_qwen35_weights.json 的完整权重目录
PINNED_REVISION = ""  # 留空则由固定环境中的 HfApi 解析并写入运行副本
# 双卡拓扑：两种方式，互斥。
#   DP（data parallel）：每张卡一份完整模型，各自独立服务请求，卡间零通信。
#   TP（tensor parallel）：一份模型切到两张卡，每次前向都要跨 PCIe all-reduce。
# T4 之间没有 NVLink，TP 每层通信会走 PCIe；本轮使用 DP2。
# 两个模型副本并行加载，Runner 每批提交 32 条、每个 replica 调度上限为 16。
PARALLEL_MODE = "dp"          # "dp" 或 "tp"
DATA_PARALLEL = 2 if PARALLEL_MODE == "dp" else 1
TENSOR_PARALLEL = 1 if PARALLEL_MODE == "dp" else 2
GPU_COUNT_REQUIRED = DATA_PARALLEL * TENSOR_PARALLEL
GPU_MEMORY_UTILIZATION = 0.80
# FlashInfer 会为 SM 7.5 现场 JIT 编译，最后一步需要链接 libcuda.so（属于 NVIDIA
# 驱动，Kaggle 容器没有 stubs），报 "cannot find -lcuda"。TRITON_ATTN 是纯 Triton
# 实现，不需要 nvcc/链接，因此作为默认值。
ATTENTION_BACKEND = "TRITON_ATTN"
# DP2: 每个 vLLM replica 最多调度 16 条；Runner 向两张卡提交全局 batch 32，
# 后端并行拆成每个 replica 16 条。TP2 时每个 engine 和 Runner batch 都是 32.
MAX_NUM_SEQS = 16 if PARALLEL_MODE == "dp" else 32
RUNNER_BATCH_SIZE = 32
# A fresh run-id prevents the earlier B32 result (which had no per-request
# TTFT samples) from being accepted as already complete on resume.
RUN_TAG = (f"dp{DATA_PARALLEL}" if PARALLEL_MODE == "dp" else f"tp{TENSOR_PARALLEL}") + f"_b{RUNNER_BATCH_SIZE}_ttft"
SMOKE_RUN_ID = f"qwen35_4b_smoke_{RUN_TAG}"
TEST_RUN_ID = f"qwen35_4b_test_{RUN_TAG}"


## 1. 验证 Qwen3.5 vLLM 包并准备独立工作目录


In [ ]:
import hmac, stat

MAX_ARCHIVE_BYTES = 512 * 1024**2
MAX_ARCHIVE_FILES = 20_000
MAX_EXPANDED_BYTES = 512 * 1024**2
MAX_MEMBER_BYTES = 16 * 1024**2
MAX_COMPRESSION_RATIO = 50.0
FREE_SPACE_RESERVE = 256 * 1024**2

if not isinstance(EXPECTED_MANIFEST_SHA256, str) or re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST_SHA256) is None:
    raise RuntimeError("set EXPECTED_MANIFEST_SHA256 from the trusted local package command")


def _safe_bundle_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name
            or path.as_posix() != name or any(ord(character) < 32 for character in name)):
        raise RuntimeError("unsafe package path: " + repr(name))
    return path


def _open_bounded_archive(bundle):
    if not bundle.is_file():
        return None, None, 0
    if bundle.stat().st_size > MAX_ARCHIVE_BYTES:
        raise RuntimeError("package exceeds compressed-size budget")
    archive = zipfile.ZipFile(bundle)
    infos = archive.infolist()
    if not infos or len(infos) > MAX_ARCHIVE_FILES:
        raise RuntimeError("package entry count is outside the allowed budget")
    index, folded, expanded = {}, set(), 0
    for info in infos:
        _safe_bundle_name(info.filename)
        folded_name = info.filename.casefold()
        if info.filename in index or folded_name in folded:
            raise RuntimeError("package contains duplicate or case-colliding paths")
        mode = (info.external_attr >> 16) & 0xFFFF
        if (info.is_dir() or info.flag_bits & 1 or stat.S_IFMT(mode) not in (0, stat.S_IFREG)
                or info.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)):
            raise RuntimeError("package contains an unsupported entry")
        if info.file_size < 0 or info.file_size > MAX_MEMBER_BYTES:
            raise RuntimeError("package member exceeds size budget")
        if info.file_size and (not info.compress_size
                or info.file_size / info.compress_size > MAX_COMPRESSION_RATIO):
            raise RuntimeError("package member exceeds compression-ratio budget")
        expanded += info.file_size
        if expanded > MAX_EXPANDED_BYTES:
            raise RuntimeError("package exceeds expanded-size budget")
        index[info.filename] = info
        folded.add(folded_name)
    return archive, index, expanded


def _directory_member(bundle, name):
    source = bundle / name
    if source.is_symlink():
        raise RuntimeError("package symlinks are unsupported: " + name)
    path = source.resolve()
    if not path.is_relative_to(bundle.resolve()) or not path.is_file():
        raise RuntimeError("unsafe package file: " + name)
    if path.stat().st_size > MAX_MEMBER_BYTES:
        raise RuntimeError("package member exceeds size budget")
    return path


def _member_bytes(bundle, archive, index, name):
    if archive:
        info = index.get(name)
        if info is None:
            raise RuntimeError("package member is missing: " + name)
        with archive.open(info) as handle:
            content = handle.read(MAX_MEMBER_BYTES + 1)
        if len(content) != info.file_size or len(content) > MAX_MEMBER_BYTES:
            raise RuntimeError("package member size changed while reading")
        return content
    return _directory_member(bundle, name).read_bytes()


def _trusted_manifest(bundle, archive, index, marker):
    raw = _member_bytes(bundle, archive, index, marker)
    digest = hashlib.sha256(raw).hexdigest()
    if not hmac.compare_digest(digest, EXPECTED_MANIFEST_SHA256):
        raise RuntimeError("package manifest is not the trusted release")
    return raw, json.loads(raw)


def _validated_bundle_entries(bundle, archive, index, marker, manifest, prefix):
    entries = manifest.get("files")
    if not isinstance(entries, list) or len(entries) > MAX_ARCHIVE_FILES - 1:
        raise RuntimeError("invalid package file list")
    expected, folded, total = {}, set(), 0
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {"path", "bytes", "sha256"}:
            raise RuntimeError("invalid package entry")
        name, size, digest = entry["path"], entry["bytes"], entry["sha256"]
        path = _safe_bundle_name(name)
        if (not name.startswith(prefix) or not isinstance(size, int) or isinstance(size, bool)
                or size < 0 or size > MAX_MEMBER_BYTES or not isinstance(digest, str)
                or re.fullmatch(r"[0-9a-f]{64}", digest) is None):
            raise RuntimeError("invalid package path, size, or digest")
        if name in expected or name.casefold() in folded:
            raise RuntimeError("package manifest contains duplicate paths")
        if archive:
            if name not in index or index[name].file_size != size:
                raise RuntimeError("package manifest differs from ZIP metadata")
        else:
            if _directory_member(bundle, name).stat().st_size != size:
                raise RuntimeError("package manifest differs from directory metadata")
        expected[name] = entry
        folded.add(name.casefold())
        total += size
        if total > MAX_EXPANDED_BYTES:
            raise RuntimeError("package manifest exceeds expanded-size budget")
    actual = set(index) if archive else {marker, *expected}
    if actual != set(expected) | {marker}:
        raise RuntimeError("package file set differs from manifest")
    disk_root = WORK
    while not disk_root.exists():
        disk_root = disk_root.parent
    if shutil.disk_usage(disk_root).free < total + FREE_SPACE_RESERVE:
        raise RuntimeError("insufficient free space for bounded extraction")
    return list(expected.values())


def _verify_entry(bundle, archive, index, entry):
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    with source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            size += len(block)
            if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                raise RuntimeError("package member exceeded manifest size")
            digest.update(block)
    if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
        raise RuntimeError("package content hash mismatch: " + entry["path"])


def _copy_entry(bundle, archive, index, entry, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".partial")
    if partial.exists():
        partial.unlink()
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    try:
        with source, partial.open("xb") as output:
            for block in iter(lambda: source.read(1024 * 1024), b""):
                size += len(block)
                if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                    raise RuntimeError("package member exceeded manifest size")
                digest.update(block)
                output.write(block)
        if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
            raise RuntimeError("package content changed while copying")
        os.replace(partial, target)
    finally:
        if partial.exists():
            partial.unlink()

PACKAGE_ID = 'cuhkx-qwen35-4b-vllm-v1'
MARKER = 'qwen35_bundle_manifest.json'

if BUNDLE_INPUT is None:
    candidates = []
    for candidate_marker in INPUT.rglob(MARKER):
        try:
            if candidate_marker.stat().st_size <= MAX_MEMBER_BYTES:
                value = json.loads(candidate_marker.read_text(encoding="utf-8"))
                if value.get('package_id') == PACKAGE_ID:
                    candidates.append(candidate_marker.parent)
        except (OSError, ValueError):
            pass
    if not candidates:
        candidates = list(INPUT.rglob('qwen35_4b.zip'))
    if len(candidates) != 1:
        raise RuntimeError(f"found {len(candidates)} matching packages; set BUNDLE_INPUT")
    BUNDLE_INPUT = candidates[0]
BUNDLE_INPUT = Path(BUNDLE_INPUT).resolve()
archive, archive_index, expanded_bytes = _open_bounded_archive(BUNDLE_INPUT)
try:
    raw_manifest, manifest = _trusted_manifest(
        BUNDLE_INPUT, archive, archive_index, MARKER)
    if manifest.get("schema_version") != 1 or manifest.get('package_id') != PACKAGE_ID:
        raise RuntimeError("wrong package identity")
    if manifest.get("inference_engine") != "vllm_0.19.1_dual_gpu":
        raise RuntimeError("package was not built for the vLLM dual-GPU lane")
    entries = _validated_bundle_entries(
        BUNDLE_INPUT, archive, archive_index, MARKER, manifest, 'qwen35_repo/')
    for entry in entries:
        _verify_entry(BUNDLE_INPUT, archive, archive_index, entry)
    MANIFEST_SHA256 = hashlib.sha256(raw_manifest).hexdigest()
    WORK_ROOT = WORK.resolve()
    RUNTIME = WORK_ROOT / ('qwen35_runtime_' + MANIFEST_SHA256[:12])
    if RUNTIME.is_symlink():
        raise RuntimeError("runtime root must not be a symlink")
    RUNTIME_ROOT = RUNTIME.resolve()
    if not RUNTIME_ROOT.is_relative_to(WORK_ROOT):
        raise RuntimeError("runtime root escapes working directory")
    for entry in entries:
        candidate = RUNTIME_ROOT / entry["path"]
        target = candidate.resolve()
        if candidate.is_symlink() or not target.is_relative_to(RUNTIME_ROOT):
            raise RuntimeError("runtime path escapes package root")
        if target.exists():
            if target.is_symlink() or not target.is_file():
                raise RuntimeError("runtime contains an unsafe existing path")
            digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if target.stat().st_size != entry["bytes"] or digest != entry["sha256"]:
                raise RuntimeError("runtime copy was modified; use a new experiment/runtime")
    for entry in entries:
        target = (RUNTIME_ROOT / entry["path"]).resolve()
        if not target.exists():
            _copy_entry(BUNDLE_INPUT, archive, archive_index, entry, target)
    REPO = RUNTIME_ROOT / 'qwen35_repo'
    (RUNTIME_ROOT / MARKER).write_bytes(raw_manifest)
finally:
    if archive:
        archive.close()
print("Verified repository:", REPO)
print("Trusted manifest SHA256:", MANIFEST_SHA256)
print("Training included:", manifest["training_included"])


## 2. 独立 Python 3.11 环境（Qwen3.5 专用，含 vLLM 0.19.1）


In [ ]:
VENV = RUNTIME / "venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    with tempfile.TemporaryDirectory(prefix="cuhkx_uv_bootstrap_", dir=WORK) as bootstrap_dir:
        BOOT = Path(bootstrap_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(BOOT), "--no-deps",
                        "--require-hashes", "--only-binary=:all:",
                        "-r", str(REPO / "requirements/bootstrap.lock.txt")], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", str(VENV)],
                       env={**os.environ, "PYTHONPATH": str(BOOT)}, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--require-hashes", "--only-binary=:all:",
                "-r", str(REPO / "requirements/qwen35.lock.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
compatibility_probe = (
    "import json, transformers, torch, vllm; "
    "print(json.dumps({'transformers': transformers.__version__, 'vllm': vllm.__version__, "
    "'torch': torch.__version__}))"
)
print(subprocess.check_output([str(PYTHON), "-c", compatibility_probe], text=True))
probe = ("import json,sys,torch; assert sys.version_info[:2]==(3,11); "
         "assert torch.cuda.is_available(), 'a cloud CUDA GPU is required'; "
         "count=torch.cuda.device_count(); "
         "assert count>=REQUIRED_GPUS, f'{PARALLEL_LABEL} needs {REQUIRED_GPUS} GPUs'; "
         "print(json.dumps({'python':sys.version,'torch':torch.__version__,"
         "'cuda':torch.version.cuda,'device_count':count,"
         "'devices':[torch.cuda.get_device_name(i) for i in range(count)]}))")
environment = subprocess.check_output(
    [str(PYTHON), "-c",
     f"REQUIRED_GPUS={GPU_COUNT_REQUIRED};PARALLEL_LABEL={PARALLEL_MODE!r};{probe}"], text=True)
(RUNTIME / "environment.json").write_text(environment, encoding="utf-8")
print(environment)
CLOUD_ENV = {**os.environ, "PYTHONPATH": str(REPO / "src"), "PYTHONDONTWRITEBYTECODE": "1",
             "PYTHONUNBUFFERED": "1", "CUHKX_TRACEBACK": "1"}

def command(*args):
    return [str(PYTHON), "-m", "cuhkx.cli", *args, "--project-root", str(REPO)]

def cloud(*args):
    result = subprocess.run(command(*args), cwd=REPO, env=CLOUD_ENV, capture_output=True,
                            text=True, encoding="utf-8", errors="replace")
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr, flush=True)
    if result.returncode:
        raise RuntimeError(f"{args[0]} exited with code {result.returncode}\n{result.stdout}\n{result.stderr}")
    return result


## 3. 固定 Qwen3.5 revision、准备权重并检查缓存


In [ ]:
if not PINNED_REVISION:
    PINNED_REVISION = subprocess.check_output(
        [str(PYTHON), "-c", "from huggingface_hub import HfApi; print(HfApi().model_info('Qwen/Qwen3.5-4B').sha)"],
        env=CLOUD_ENV, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", PINNED_REVISION):
    raise RuntimeError("Qwen3.5 revision 不是 40 位小写 commit SHA")
import yaml
profile = REPO / "configs/qwen35_4b.yaml"
profile_value = yaml.safe_load(profile.read_text(encoding="utf-8"))
old_revision = profile_value["model"].get("revision")
if old_revision not in (None, PINNED_REVISION):
    raise RuntimeError("包内 Qwen3.5 revision 与本次指定版本不同")
profile_value["model"]["revision"] = PINNED_REVISION
profile.write_text(yaml.safe_dump(profile_value, sort_keys=False), encoding="utf-8")
if WEIGHTS_INPUT is None:
    candidates = []
    for receipt in INPUT.rglob("cuhkx_qwen35_weights.json"):
        try:
            value = json.loads(receipt.read_text(encoding="utf-8"))
            if value.get("model_id") == "Qwen/Qwen3.5-4B" and value.get("revision") == PINNED_REVISION:
                candidates.append(receipt.parent)
        except (OSError, ValueError):
            continue
    if len(candidates) > 1:
        raise RuntimeError("找到多份 Qwen3.5 权重，请手动设置 WEIGHTS_INPUT")
    WEIGHTS = candidates[0] if candidates else Path("/tmp") / ("qwen35_weights_" + PINNED_REVISION[:12])
else:
    WEIGHTS = Path(WEIGHTS_INPUT)
cloud("fetch-qwen35-weights", "--weights-dir", str(WEIGHTS), "--revision", PINNED_REVISION)
cloud("check", "--profile", "qwen35", "--dataset", "test")
cloud("check", "--profile", "qwen35", "--dataset", "pilot")
print("Qwen3.5 weights:", WEIGHTS)
print("Qwen3.5 revision:", PINNED_REVISION)


## 4. vLLM 双卡 smoke：test 前 16 QA，验证拓扑、答案约束与前缀缓存命中率


In [ ]:
VLLM = ["--backend", "vllm",
        "--tensor-parallel-size", str(TENSOR_PARALLEL),
        "--data-parallel-size", str(DATA_PARALLEL),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--attention-backend", ATTENTION_BACKEND,
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--runner-batch-size", str(RUNNER_BATCH_SIZE)]
cloud("predict", "--profile", "qwen35", *VLLM, "--dataset", "test", "--limit", "16",
      "--run-id", SMOKE_RUN_ID, "--weights-dir", str(WEIGHTS), "--resume")
cloud("verify-run", "--profile", "qwen35", "--run-id", SMOKE_RUN_ID)
smoke = json.loads((REPO / "outputs" / SMOKE_RUN_ID / "run_summary.json").read_text())
backend_metadata = smoke["backend"]
if backend_metadata.get("backend") != "qwen35_4b_vllm":
    raise RuntimeError("smoke run did not use the vLLM engine")
if backend_metadata.get("tensor_parallel_size") != TENSOR_PARALLEL:
    raise RuntimeError(f"vLLM ran with TP={backend_metadata.get('tensor_parallel_size')}, expected {TENSOR_PARALLEL}")
if PARALLEL_MODE == "dp" and backend_metadata.get("data_parallel_size") != DATA_PARALLEL:
    raise RuntimeError(f"vLLM ran with DP={backend_metadata.get('data_parallel_size')}, expected {DATA_PARALLEL}")
resolved = json.loads((REPO / "outputs" / SMOKE_RUN_ID / "resolved_config.json").read_text())
options = resolved["contract"]["engine_options"]
if options.get("max_num_seqs") != MAX_NUM_SEQS:
    raise RuntimeError(f"engine max_num_seqs={options.get('max_num_seqs')}, expected {MAX_NUM_SEQS}")
if options.get("runner_batch_size") != RUNNER_BATCH_SIZE:
    raise RuntimeError(f"runner_batch_size={options.get('runner_batch_size')}, expected {RUNNER_BATCH_SIZE}")
if backend_metadata.get("attention_backend") != ATTENTION_BACKEND:
    raise RuntimeError(f"vLLM ran with attention backend {backend_metadata.get('attention_backend')}, expected {ATTENTION_BACKEND}")
# Worker 显存遥测必须真的采到数据：序列化失败时 workers 为空、error 非空。
workers = backend_metadata.get("workers") or {}
if not workers.get("workers"):
    raise RuntimeError(f"worker memory telemetry is empty: {workers.get('error')}")
# 前缀缓存命中率：能测到就打印；这个 vLLM 构建不上报该字段时只告警，
# 并把 metrics_shape 打出来，让下一次运行自己说明字段名。
prefix = backend_metadata.get("prefix_cache") or {}
if prefix.get("reported_requests"):
    print("prefix cache hit rate:", prefix)
else:
    print("WARNING: prefix cache counters were not reported by this engine build:",
          prefix)
    print("engine metrics field inventory:",
          json.dumps(backend_metadata.get("metrics_shape"), indent=2, default=str))
print(json.dumps({"backend": backend_metadata, "prefix_cache": prefix}, indent=2))
ttft_summary = smoke.get("latency", {}).get("ttft_ms", {})
print("per-request engine TTFT summary (ms):",
      json.dumps({key: value for key, value in ttft_summary.items() if key != "samples"}, indent=2))
print("sampled device memory peak (MiB):",
      json.dumps(smoke.get("gpu_memory", {}).get("peaks", {}).get("device_polling", {}), indent=2))


## 5. 完整 test：682 QA 与提交文件（vLLM）


In [ ]:
cloud("predict", "--profile", "qwen35", *VLLM, "--dataset", "test",
      "--run-id", TEST_RUN_ID, "--weights-dir", str(WEIGHTS), "--resume")
cloud("verify-run", "--profile", "qwen35", "--run-id", TEST_RUN_ID)
cloud("submit", "--profile", "qwen35", "--run-id", TEST_RUN_ID)
print("Qwen3.5 submission:", REPO / "outputs" / TEST_RUN_ID / "submission.csv")
print("Qwen3.5 run evidence:", REPO / "outputs" / TEST_RUN_ID)
test_summary = json.loads((REPO / "outputs" / TEST_RUN_ID / "run_summary.json").read_text())
ttft_summary = test_summary.get("latency", {}).get("ttft_ms", {})
print("per-request engine TTFT summary (ms):",
      json.dumps({key: value for key, value in ttft_summary.items() if key != "samples"}, indent=2))
print("sampled device memory peak (MiB):",
      json.dumps(test_summary.get("gpu_memory", {}).get("peaks", {}).get("device_polling", {}), indent=2))
